# Scaffold Results

Recovered workflow. See `docs/WORKFLOW_ORDER.md` for dependencies and execution restrictions. Source cell numbers below are zero-based.


Source cell: 65.


In [ ]:
# ============================================================
# FINAL CORRECTED SCAFFOLD — 3-SEED AGGREGATION
#
# NO TRAINING
#
# Reads:
#   Linear MLP seeds 42,43,44
#   Graph GCN seeds 42,43,44
#
# Produces:
#   - all_seed_results.csv
#   - mean_sd_summary.csv
#   - matched_seed_differences.csv
#   - paper_ready_table.csv
#   - FINAL_SCAFFOLD_3SEED_RESULTS.xlsx
#   - final_scaffold_summary.json
#   - README.txt
#   - ZIP package
# ============================================================

import json
import shutil
import hashlib
from pathlib import Path

import numpy as np
import pandas as pd
from IPython.display import display, FileLink


# ============================================================
# 1) PATHS
# ============================================================

PROTOCOL = Path(
    "/kaggle/working/"
    "DDI_SCAFFOLD_NONCHIRAL_RECONSTRUCTED_20260921_190107_430125"
)

RUN_ROOT = (
    PROTOCOL
    / "controlled_rerun_v1"
)

OUT = (
    PROTOCOL
    / "FINAL_SCAFFOLD_3SEED_SUMMARY"
)

if OUT.exists():
    shutil.rmtree(OUT)

OUT.mkdir(
    parents=True,
    exist_ok=True,
)

SEEDS = [42, 43, 44]
MODELS = ["linear_mlp", "graph"]


# ============================================================
# 2) HELPERS
# ============================================================

def sha256_file(path):
    digest = hashlib.sha256()

    with path.open("rb") as handle:
        for block in iter(
            lambda: handle.read(1024 * 1024),
            b"",
        ):
            digest.update(block)

    return digest.hexdigest()


def mean_sd(series):
    values = pd.to_numeric(
        series,
        errors="raise",
    ).to_numpy(dtype=float)

    return (
        float(values.mean()),
        float(values.std(ddof=1)),
    )


# ============================================================
# 3) READ ALL SIX FINAL RESULTS
# ============================================================

rows = []

source_manifest = []

for model in MODELS:

    for seed in SEEDS:

        run_dir = (
            RUN_ROOT
            / model
            / f"seed_{seed}"
        )

        result_json = (
            run_dir
            / "final_result.json"
        )

        result_csv = (
            run_dir
            / "final_result.csv"
        )

        prediction_file = (
            run_dir
            / "test_predictions.csv"
        )

        checkpoint_file = (
            run_dir
            / "best_checkpoint.pt"
        )

        assert result_json.is_file(), (
            f"Missing result:\n{result_json}"
        )

        assert result_csv.is_file(), (
            f"Missing result CSV:\n{result_csv}"
        )

        assert prediction_file.is_file(), (
            f"Missing predictions:\n{prediction_file}"
        )

        assert checkpoint_file.is_file(), (
            f"Missing best checkpoint:\n{checkpoint_file}"
        )


        result = json.loads(
            result_json.read_text(
                encoding="utf-8"
            )
        )


        # ====================================================
        # STRICT RUN QA
        # ====================================================

        assert result["model"] == model, (
            model,
            seed,
            result["model"],
        )

        assert result["seed"] == seed

        assert (
            result["protocol"]
            ==
            "both_nonchiral_scaffolds_unseen"
        )

        assert result["test_rows"] == 11142

        assert result["test_observed_classes"] == 56

        assert result["checkpoint_gap"] <= 1e-12


        # ====================================================
        # HASH QA
        # ====================================================

        actual_checkpoint_hash = sha256_file(
            checkpoint_file
        )

        actual_prediction_hash = sha256_file(
            prediction_file
        )

        assert (
            actual_checkpoint_hash
            ==
            result["best_checkpoint_sha256"]
        ), (
            f"Checkpoint hash mismatch: "
            f"{model} seed {seed}"
        )

        assert (
            actual_prediction_hash
            ==
            result["test_predictions_sha256"]
        ), (
            f"Prediction hash mismatch: "
            f"{model} seed {seed}"
        )


        # ====================================================
        # PREDICTION ROW QA
        # ====================================================

        pred = pd.read_csv(
            prediction_file
        )

        assert len(pred) == 11142

        assert "pred_label_id" in pred.columns
        assert "correct" in pred.columns


        # ====================================================
        # RECORD
        # ====================================================

        rows.append(
            result
        )

        source_manifest.append(
            {
                "model": model,
                "seed": seed,
                "final_result_json":
                    str(result_json),
                "final_result_json_sha256":
                    sha256_file(result_json),
                "best_checkpoint_sha256":
                    actual_checkpoint_hash,
                "test_predictions_sha256":
                    actual_prediction_hash,
            }
        )


all_results = pd.DataFrame(
    rows
).sort_values(
    ["model", "seed"]
).reset_index(drop=True)


assert len(all_results) == 6

assert set(
    all_results["seed"]
) == {42, 43, 44}

assert (
    all_results.groupby("model")["seed"]
    .nunique()
    .eq(3)
    .all()
)


print("=" * 90)
print("ALL SIX SCAFFOLD RUNS VERIFIED")
print("=" * 90)

display(
    all_results[
        [
            "model",
            "seed",
            "completed_epochs",
            "best_epoch",
            "cal_macro83",
            "test_accuracy",
            "test_macro56",
            "test_macro84",
            "test_weighted_f1",
        ]
    ]
)


# ============================================================
# 4) SAVE RAW ALL-SEED RESULTS
# ============================================================

all_results_path = (
    OUT
    / "all_seed_results.csv"
)

all_results.to_csv(
    all_results_path,
    index=False,
)


manifest = pd.DataFrame(
    source_manifest
)

manifest.to_csv(
    OUT / "source_manifest.csv",
    index=False,
)


# ============================================================
# 5) MEAN ± SAMPLE SD
# ============================================================

metrics = [
    "cal_macro83",
    "test_accuracy",
    "test_macro56",
    "test_macro84",
    "test_weighted_f1",
]

summary_rows = []

for model in MODELS:

    part = all_results[
        all_results["model"] == model
    ].copy()

    row = {
        "model": model,
        "n_seeds": len(part),
        "seeds": "42,43,44",
    }

    for metric in metrics:

        mean_value, sd_value = mean_sd(
            part[metric]
        )

        row[f"{metric}_mean"] = mean_value
        row[f"{metric}_sd"] = sd_value

        row[
            f"{metric}_mean_percent"
        ] = mean_value * 100

        row[
            f"{metric}_sd_percent"
        ] = sd_value * 100

        row[
            f"{metric}_paper"
        ] = (
            f"{mean_value * 100:.4f} "
            f"± {sd_value * 100:.4f}%"
        )

    row["best_epoch_mean"] = float(
        part["best_epoch"].mean()
    )

    row["completed_epochs_mean"] = float(
        part["completed_epochs"].mean()
    )

    summary_rows.append(row)


summary = pd.DataFrame(
    summary_rows
)

summary_path = (
    OUT
    / "mean_sd_summary.csv"
)

summary.to_csv(
    summary_path,
    index=False,
)


print("\n" + "=" * 90)
print("MEAN ± SAMPLE SD")
print("=" * 90)

display(
    summary[
        [
            "model",
            "cal_macro83_paper",
            "test_accuracy_paper",
            "test_macro56_paper",
            "test_macro84_paper",
            "test_weighted_f1_paper",
        ]
    ]
)


# ============================================================
# 6) MATCHED-SEED MLP - GRAPH DIFFERENCES
# ============================================================

mlp = (
    all_results[
        all_results["model"]
        ==
        "linear_mlp"
    ]
    .set_index("seed")
)

graph = (
    all_results[
        all_results["model"]
        ==
        "graph"
    ]
    .set_index("seed")
)


assert list(
    mlp.index
) == [42, 43, 44]

assert list(
    graph.index
) == [42, 43, 44]


difference_rows = []

comparison_metrics = [
    "test_accuracy",
    "test_macro56",
    "test_macro84",
    "test_weighted_f1",
]

for seed in SEEDS:

    row = {
        "seed": seed
    }

    for metric in comparison_metrics:

        mlp_value = float(
            mlp.loc[seed, metric]
        )

        graph_value = float(
            graph.loc[seed, metric]
        )

        diff = (
            mlp_value
            -
            graph_value
        )

        row[
            f"mlp_{metric}"
        ] = mlp_value

        row[
            f"graph_{metric}"
        ] = graph_value

        row[
            f"mlp_minus_graph_{metric}"
        ] = diff

        row[
            f"mlp_minus_graph_{metric}_pp"
        ] = diff * 100

    difference_rows.append(row)


matched = pd.DataFrame(
    difference_rows
)


# ------------------------------------------------------------
# Add mean / sample SD of matched differences
# ------------------------------------------------------------

mean_diff_row = {
    "seed": "Mean"
}

sd_diff_row = {
    "seed": "Sample SD"
}

for metric in comparison_metrics:

    col = (
        f"mlp_minus_graph_{metric}"
    )

    pp_col = (
        f"mlp_minus_graph_{metric}_pp"
    )

    mean_diff_row[col] = float(
        matched[col].mean()
    )

    sd_diff_row[col] = float(
        matched[col].std(ddof=1)
    )

    mean_diff_row[pp_col] = float(
        matched[pp_col].mean()
    )

    sd_diff_row[pp_col] = float(
        matched[pp_col].std(ddof=1)
    )


matched_with_summary = pd.concat(
    [
        matched,
        pd.DataFrame(
            [
                mean_diff_row,
                sd_diff_row,
            ]
        ),
    ],
    ignore_index=True,
)


matched_path = (
    OUT
    / "matched_seed_differences.csv"
)

matched_with_summary.to_csv(
    matched_path,
    index=False,
)


print("\n" + "=" * 90)
print("MATCHED-SEED DIFFERENCE: MLP − GRAPH")
print("=" * 90)

display(
    matched_with_summary
)


# ============================================================
# 7) PAPER-READY TABLE
# ============================================================

paper_rows = []

model_display = {
    "linear_mlp":
        "Linear MLP",

    "graph":
        "Graph GCN",
}


for _, row in summary.iterrows():

    paper_rows.append(
        {
            "Model":
                model_display[
                    row["model"]
                ],

            "Seeds":
                "42, 43, 44",

            "Calibration Macro-F1 (83)":
                row[
                    "cal_macro83_paper"
                ],

            "Test Accuracy":
                row[
                    "test_accuracy_paper"
                ],

            "Test Macro-F1 (56 observed)":
                row[
                    "test_macro56_paper"
                ],

            "Test Macro-F1 (84-class space)":
                row[
                    "test_macro84_paper"
                ],

            "Test Weighted-F1":
                row[
                    "test_weighted_f1_paper"
                ],
        }
    )


paper_table = pd.DataFrame(
    paper_rows
)


paper_table.to_csv(
    OUT
    / "paper_ready_table.csv",
    index=False,
)


print("\n" + "=" * 90)
print("PAPER-READY TABLE")
print("=" * 90)

display(
    paper_table
)


# ============================================================
# 8) PER-SEED PAPER TABLE
# ============================================================

per_seed_paper = all_results[
    [
        "model",
        "seed",
        "best_epoch",
        "completed_epochs",
        "cal_macro83",
        "test_accuracy",
        "test_macro56",
        "test_macro84",
        "test_weighted_f1",
    ]
].copy()


for col in [
    "cal_macro83",
    "test_accuracy",
    "test_macro56",
    "test_macro84",
    "test_weighted_f1",
]:
    per_seed_paper[col] *= 100


per_seed_paper["model"] = (
    per_seed_paper["model"]
    .map(model_display)
)


per_seed_paper = per_seed_paper.rename(
    columns={
        "model":
            "Model",

        "seed":
            "Seed",

        "best_epoch":
            "Best Epoch",

        "completed_epochs":
            "Completed Epochs",

        "cal_macro83":
            "Calibration Macro-F1 (83) %",

        "test_accuracy":
            "Test Accuracy %",

        "test_macro56":
            "Test Macro-F1 (56) %",

        "test_macro84":
            "Test Macro-F1 (84) %",

        "test_weighted_f1":
            "Test Weighted-F1 %",
    }
)


per_seed_paper.to_csv(
    OUT
    / "per_seed_paper_table.csv",
    index=False,
)


# ============================================================
# 9) CALIBRATION -> TEST DROP
# ============================================================

drop_rows = []

for _, row in all_results.iterrows():

    drop_rows.append(
        {
            "model":
                row["model"],

            "seed":
                int(row["seed"]),

            "cal_macro83":
                row["cal_macro83"],

            "test_macro56":
                row["test_macro56"],

            "cal_to_test_macro_drop":
                (
                    row["cal_macro83"]
                    -
                    row["test_macro56"]
                ),

            "cal_to_test_macro_drop_pp":
                (
                    row["cal_macro83"]
                    -
                    row["test_macro56"]
                ) * 100,
        }
    )


drop_df = pd.DataFrame(
    drop_rows
)


drop_df.to_csv(
    OUT
    / "calibration_to_scaffold_test_drop.csv",
    index=False,
)


# ============================================================
# 10) FINAL JSON SUMMARY
# ============================================================

def get_summary_value(
    model,
    metric,
    stat,
):
    row = summary[
        summary["model"] == model
    ].iloc[0]

    return float(
        row[
            f"{metric}_{stat}"
        ]
    )


final_summary = {
    "protocol":
        "both_nonchiral_scaffolds_unseen",

    "seeds":
        [42, 43, 44],

    "n_seeds":
        3,

    "test_rows_per_seed":
        11142,

    "test_observed_classes":
        56,

    "total_class_space":
        84,

    "statistics":
        "Arithmetic mean and sample standard deviation (ddof=1)",

    "linear_mlp": {
        metric: {
            "mean":
                get_summary_value(
                    "linear_mlp",
                    metric,
                    "mean",
                ),

            "sample_sd":
                get_summary_value(
                    "linear_mlp",
                    metric,
                    "sd",
                ),
        }
        for metric in metrics
    },

    "graph": {
        metric: {
            "mean":
                get_summary_value(
                    "graph",
                    metric,
                    "mean",
                ),

            "sample_sd":
                get_summary_value(
                    "graph",
                    metric,
                    "sd",
                ),
        }
        for metric in metrics
    },

    "interpretation_note": (
        "Under this controlled both-nonchiral-scaffolds-unseen "
        "protocol, the topology-free Linear MLP achieved higher "
        "held-out scaffold test metrics than the Graph GCN across "
        "all three matched seeds. This is protocol-specific and "
        "should not be generalized beyond this evaluation setting."
    ),
}


with (
    OUT
    / "final_scaffold_summary.json"
).open(
    "w",
    encoding="utf-8",
) as handle:

    json.dump(
        final_summary,
        handle,
        indent=2,
        ensure_ascii=False,
    )


# ============================================================
# 11) README
# ============================================================

README = f"""
FINAL CORRECTED NONCHIRAL SCAFFOLD — 3-SEED SUMMARY
===================================================

Protocol:
both_nonchiral_scaffolds_unseen

Seeds:
42, 43, 44

Models:
1. Linear MLP
2. Graph GCN

Test rows:
11,142 per seed

Observed classes in scaffold test:
56

Full modeled class space:
84

Statistics:
Arithmetic mean ± sample standard deviation (n=3, ddof=1).

IMPORTANT:
- This script performs NO training.
- All metrics are read from completed final_result.json files.
- Checkpoint and test-prediction SHA256 hashes are revalidated.
- The comparison is matched by seed.
- Macro-F1(56) is computed over the classes observed in the scaffold test.
- Macro-F1(84) retains the full 84-class modeled output space.
- Results are specific to the corrected both-nonchiral-scaffolds-unseen protocol.

PAPER INTERPRETATION:
The corrected scaffold-disjoint evaluation produced a marked decrease
relative to the calibration performance for both models. Across the
three matched seeds, the topology-free Linear MLP retained substantially
higher held-out scaffold performance than the Graph GCN. This result
should be interpreted specifically within the strict scaffold-disjoint
protocol rather than as a general ranking of graph and non-graph models.

Files:
- all_seed_results.csv
- mean_sd_summary.csv
- matched_seed_differences.csv
- per_seed_paper_table.csv
- paper_ready_table.csv
- calibration_to_scaffold_test_drop.csv
- source_manifest.csv
- final_scaffold_summary.json
- FINAL_SCAFFOLD_3SEED_RESULTS.xlsx
""".strip()


(
    OUT
    / "README.txt"
).write_text(
    README,
    encoding="utf-8",
)


# ============================================================
# 12) EXCEL WORKBOOK
# ============================================================

excel_path = (
    OUT
    / "FINAL_SCAFFOLD_3SEED_RESULTS.xlsx"
)


with pd.ExcelWriter(
    excel_path,
    engine="openpyxl",
) as writer:

    paper_table.to_excel(
        writer,
        sheet_name="Paper Ready Summary",
        index=False,
    )

    per_seed_paper.to_excel(
        writer,
        sheet_name="Per Seed Results",
        index=False,
    )

    summary.to_excel(
        writer,
        sheet_name="Mean SD Full",
        index=False,
    )

    matched_with_summary.to_excel(
        writer,
        sheet_name="Matched Differences",
        index=False,
    )

    drop_df.to_excel(
        writer,
        sheet_name="Calibration Test Drop",
        index=False,
    )

    all_results.to_excel(
        writer,
        sheet_name="Raw Results",
        index=False,
    )

    manifest.to_excel(
        writer,
        sheet_name="Source Manifest",
        index=False,
    )


# ============================================================
# 13) ZIP FINAL PACKAGE
# ============================================================

zip_base = (
    PROTOCOL
    / "FINAL_SCAFFOLD_3SEED_SUMMARY"
)

zip_path = shutil.make_archive(
    str(zip_base),
    "zip",
    root_dir=OUT,
)


# ZIP integrity
import zipfile

with zipfile.ZipFile(
    zip_path,
    "r",
) as archive:

    assert archive.testzip() is None


# ============================================================
# 14) FINAL PRINT
# ============================================================

print("\n" + "=" * 90)
print(" FINAL SCAFFOLD 3-SEED ANALYSIS COMPLETE")
print("=" * 90)

for model in [
    "linear_mlp",
    "graph",
]:

    r = summary[
        summary["model"] == model
    ].iloc[0]

    print(
        f"\n{model.upper()}"
    )

    print(
        "Accuracy:",
        r["test_accuracy_paper"],
    )

    print(
        "Macro-F1 (56 observed):",
        r["test_macro56_paper"],
    )

    print(
        "Macro-F1 (84 space):",
        r["test_macro84_paper"],
    )

    print(
        "Weighted-F1:",
        r["test_weighted_f1_paper"],
    )


print("\nFiles saved to:")
print(OUT)

print("\nZIP:")
print(zip_path)

print("\n No training performed.")
print(" Six completed runs verified.")
print(" SHA256 checks passed.")
print(" Mean ± sample SD calculated.")
print(" Matched-seed comparison calculated.")
print(" Excel + CSV + JSON + ZIP created.")

display(
    FileLink(
        str(
            Path(zip_path).relative_to(
                "/kaggle/working"
            )
        )
    )
)